# Random Forest habitat-type model

Train a multiclass Random Forest to predict the monitoring-plot `habitat_code` from Sentinel-1/2 index summaries. The notebook uses the spatial `train`/`test` labels created in `data_preparation_3.ipynb`; it does not create a new random split. Raster index values are already converted from stored integers using the catalog scale factor in `raster_data_extraction_2.ipynb`, so no min-max scaling is applied here.

Run the preparation and extraction notebooks first so `working/obsevation_plots_with_year_indices_split.gpkg` exists.

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import make_pipeline

DATA_PATH = Path("working/obsevation_plots_with_year_indices_split.gpkg")
OUTPUT_DIR = Path("working")
TARGET = "habitat_code"
SPLIT_COLUMN = "split"

indices = [
    "ndvi", "evi", "lai", "savi", "fvc", "gndvi", "ndwi", "bsi",
    "ndmi", "alpha_diversity", "rvi",
]
predictors = [
    f"{index}_{statistic}"
    for index in indices
    for statistic in ("mean", "median")
]

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Run data_preparation_1.ipynb, "
        "raster_data_extraction_2.ipynb, and data_preparation_3.ipynb first."
    )

plots = gpd.read_file(DATA_PATH)
required_columns = [TARGET, SPLIT_COLUMN, *predictors]
missing_columns = sorted(set(required_columns) - set(plots.columns))
if missing_columns:
    raise ValueError(f"Model input is missing required columns: {missing_columns}")

print(f"Input plots: {len(plots):,}")
print("Rows by split:")
display(plots[SPLIT_COLUMN].value_counts(dropna=False))
print("Missing predictor values by split:")
display(plots.groupby(SPLIT_COLUMN)[predictors].apply(lambda frame: frame.isna().sum()).T)
print("Target counts by split:")
display(pd.crosstab(plots[TARGET], plots[SPLIT_COLUMN], dropna=False))

Input plots: 1,553
Rows by split:


split
train    1237
test      316
Name: count, dtype: int64

Missing predictor values by split:


split,test,train
ndvi_mean,0,0
ndvi_median,0,0
evi_mean,0,0
evi_median,0,0
lai_mean,0,0
lai_median,0,0
savi_mean,0,0
savi_median,0,0
fvc_mean,0,0
fvc_median,0,0


Target counts by split:


split,test,train
habitat_code,,
0,34,134
2180,23,75
9010*,60,238
9020*,18,84
9050,27,101
9060,30,86
9080,39,139
9180*,11,72
91D0*,30,121


In [2]:
# Keep the ten forest habitat codes. The source data also contains "0 elupaigatüüp",
# which is not one of the habitat classes, plus a small number of unlabeled plots.
habitat_classes = {
    "2180", "9010*", "9020*", "9050", "9060", "9080",
    "91D0*", "91E0*", "91F0", "9180*",
}
plots[TARGET] = plots[TARGET].astype("string").str.strip()
valid_label = plots[TARGET].isin(habitat_classes)
print(f"Excluded unlabeled/non-target plots: {(~valid_label).sum():,}")

# Use the precomputed spatial split; do not make a new random split.
labeled = plots.loc[
    plots[SPLIT_COLUMN].isin(["train", "test"])
    & valid_label
].copy()

train = labeled.loc[labeled[SPLIT_COLUMN] == "train"]
test = labeled.loc[labeled[SPLIT_COLUMN] == "test"]
if train.empty or test.empty:
    raise ValueError("Both train and test sets must contain labeled plots.")

X_train = train[predictors]
y_train = train[TARGET]
X_test = test[predictors]
y_test = test[TARGET]

unseen_test_classes = sorted(set(y_test) - set(y_train))
if unseen_test_classes:
    print("Warning: these test classes are absent from training and cannot be predicted:", unseen_test_classes)

print(f"Training plots: {len(train):,}; test plots: {len(test):,}")
print(f"Training classes: {y_train.nunique()}; test classes: {y_test.nunique()}")
print(f"Predictors: {len(predictors)} index mean/median summaries")

Excluded unlabeled/non-target plots: 172
Training plots: 1,101; test plots: 280
Training classes: 10; test classes: 10
Predictors: 22 index mean/median summaries


In [3]:
# Imputation is fitted inside the pipeline on training data only.
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = pd.Series({
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
})
display(metrics.to_frame("test_score"))

class_labels = sorted(set(y_train) | set(y_test))
print(classification_report(
    y_test,
    y_pred,
    labels=class_labels,
    zero_division=0,
    digits=3,
))

confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=class_labels),
    index=pd.Index(class_labels, name="actual"),
    columns=pd.Index(class_labels, name="predicted"),
)
display(confusion)

,test_score
accuracy,0.475000
balanced_accuracy,0.440597
macro_f1,0.437922


              precision    recall  f1-score   support

        2180      0.625     0.652     0.638        23
       9010*      0.552     0.617     0.583        60
       9020*      0.412     0.389     0.400        18
        9050      0.407     0.407     0.407        27
        9060      0.500     0.167     0.250        30
        9080      0.500     0.590     0.541        39
       9180*      0.364     0.364     0.364        11
       91D0*      0.680     0.567     0.618        30
       91E0*      0.227     0.294     0.256        17
        91F0      0.290     0.360     0.321        25

    accuracy                          0.475       280
   macro avg      0.456     0.441     0.438       280
weighted avg      0.486     0.475     0.469       280



predicted,2180,9010*,9020*,9050,9060,9080,9180*,91D0*,91E0*,91F0
actual,,,,,,,,,,
2180,15,3,0,0,0,0,2,3,0,0
9010*,5,37,0,5,2,6,0,4,0,1
9020*,0,0,7,0,0,2,1,0,4,4
9050,0,3,2,11,2,5,1,0,1,2
9060,0,10,0,7,5,3,0,1,2,2
9080,0,4,0,1,0,23,1,0,5,5
9180*,0,0,3,1,0,0,4,0,1,2
91D0*,4,8,0,0,0,1,0,17,0,0
91E0*,0,1,2,1,1,1,0,0,5,6


In [4]:
# Inspect impurity-based importance as a first diagnostic, not as causal attribution.
forest = model.named_steps["randomforestclassifier"]
feature_importance = (
    pd.Series(forest.feature_importances_, index=predictors, name="importance")
    .sort_values(ascending=False)
)
display(feature_importance.head(20).to_frame())

# Save the fitted pipeline and held-out predictions for later inspection.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
import joblib

joblib.dump(model, OUTPUT_DIR / "random_forest_habitat_code.joblib")
predictions = pd.DataFrame({
    "row_index": test.index,
    "actual_habitat_code": y_test.to_numpy(),
    "predicted_habitat_code": y_pred,
})
predictions.to_csv(
    OUTPUT_DIR / "random_forest_habitat_code_test_predictions.csv",
    index=False,
)
print("Saved fitted model and test predictions under working/.")

,importance
rvi_mean,0.074428
alpha_diversity_mean,0.067248
rvi_median,0.066616
alpha_diversity_median,0.063206
savi_median,0.054852
evi_mean,0.052185
lai_median,0.051995
savi_mean,0.051248
lai_mean,0.051090
evi_median,0.049915


Saved fitted model and test predictions under working/.
